# Starter 2: Rotating a qubit

| | |
|---|---|
| **Level** | Introductory |
| **Time** | About 30 minutes |
| **Prerequisites** | Qubits, the Bloch sphere, rotation gates |
| **Default device** | IQM Garnet |
| **Also runs on** | Rigetti Cepheus-1-108Q, AQT IBEX Q1 |
| **Qubits** | 9 |
| **Two-qubit gates** | None |
| **Hardware jobs** | 1 |
| **Approximate cost** | Garnet at 500 shots: about 103 credits. Rigetti: about 10 credits (billed by execution time). |
| **Suggested hand-in** | Your plot, the fitted values of A and B, and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST starter series from qBraid. You may copy, edit and adapt this notebook for your course.*

The gate $R_y(\theta)$ rotates a qubit around the y-axis of the Bloch sphere. Starting from $|0\rangle$, the probability of measuring 1 is

$$P(1) = \sin^2(\theta/2).$$

At $\theta = 0$ the qubit stays in $|0\rangle$. At $\theta = \pi$ it is in $|1\rangle$. In between, the measurement outcome is random with a probability you set.

In this notebook you sweep $\theta$ from 0 to $2\pi$ and compare the measured curve with the prediction. Each angle runs on its own qubit, so the whole sweep is one hardware job.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "aws:iqm:qpu:garnet"   # device list and prices: see the README
SHOTS = 500                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

Nine angles from 0 to $2\pi$, one per qubit.

In [ ]:
thetas = np.linspace(0, 2 * np.pi, 9)
N_QUBITS = len(thetas)

qc = QuantumCircuit(N_QUBITS)
for qubit, theta in enumerate(thetas):
    qc.ry(theta, qubit)
qc.measure_all()

qc.draw(output="text")

## 2. Ideal simulation

The simulator is exact apart from shot noise: with a finite number of shots, a probability of 0.5 will come out as 0.49 or 0.52.

In [ ]:
simulator = AerSimulator()
ideal_counts = simulator.run(qc, shots=SHOTS).result().get_counts()
ideal_p1 = [prob_one(ideal_counts, q, N_QUBITS) for q in range(N_QUBITS)]

for theta, p in zip(thetas, ideal_p1):
    print(f"theta = {theta:5.2f}   P(1) = {p:.3f}   prediction = {np.sin(theta / 2) ** 2:.3f}")

## 3. Run on hardware

In [ ]:
N_JOBS = 1

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    job = device.run(qc, shots=SHOTS)
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = job.result().data.get_counts()
    print(f"Received {sum(hw_counts.values())} shots.")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

A convenient way to summarise the hardware curve is to fit

$$P(1) = A \sin^2(\theta/2) + B.$$

A perfect device gives $A = 1$ (full contrast) and $B = 0$ (no offset). Noise usually makes $A$ smaller and $B$ larger, so the curve no longer reaches 0 and 1.

In [ ]:
def fit_contrast(p1):
    basis = np.column_stack([np.sin(thetas / 2) ** 2, np.ones_like(thetas)])
    (A, B), *_ = np.linalg.lstsq(basis, np.array(p1), rcond=None)
    return A, B

fine = np.linspace(0, 2 * np.pi, 200)
plt.plot(fine, np.sin(fine / 2) ** 2, color="gray", label="prediction")
plt.plot(thetas, ideal_p1, "o", color="black", label="ideal simulation")

A, B = fit_contrast(ideal_p1)
print(f"ideal simulation: A = {A:.3f}, B = {B:.3f}")

if hw_counts:
    hw_p1 = [prob_one(hw_counts, q, N_QUBITS) for q in range(N_QUBITS)]
    A, B = fit_contrast(hw_p1)
    print(f"hardware:         A = {A:.3f}, B = {B:.3f}")
    plt.plot(thetas, hw_p1, "s", color="tab:orange", label=f"hardware ({DEVICE_ID})")

plt.xlabel("rotation angle theta")
plt.ylabel("P(1)")
plt.legend()
plt.show()

## Questions to try

1. What are $A$ and $B$ on hardware? Which effect from Starter 1 could produce $B > 0$?
2. The hardware points do not all lie on one smooth curve. Each angle ran on a different physical qubit. How could you tell whether the scatter comes from differences between qubits or from shot noise?
3. Replace `qc.ry` with `qc.rx`. What changes in the result, and why?
4. Run the same sweep on Rigetti Cepheus. Which device has the larger contrast $A$?